**Checking for the solvent**

In [1]:
import pandas as pd
df_full = pd.read_csv("beard_uvvis_cleaned.csv")  # Phase 1 output, still has solvent column
df_solvent_known = df_full[df_full["solvent"].notna()].copy()

print(f"Compounds with known solvent: {df_solvent_known.shape[0]}")
print(df_solvent_known["solvent"].value_counts().head(20))

Compounds with known solvent: 2218
solvent
DMSO               205
dichloromethane    204
THF                167
chloroform         128
DMF                115
MeOH                88
Toluene             87
CH 2 Cl 2           84
CH 2 Cl             81
ethanol             74
methanol            63
Cyclohexane         59
DCM                 54
Methanol            53
EtOH                52
acetonitrile        46
Chloroform          46
acetone             38
EtOAc               37
Ethanol             31
Name: count, dtype: int64


**Build a mapping to consolidate known variants**

In [2]:
solvent_raw_counts = df_solvent_known["solvent"].value_counts()
print(solvent_raw_counts)

solvent
DMSO               205
dichloromethane    204
THF                167
chloroform         128
DMF                115
                  ... 
Ethyl acetate        1
MCH                  1
MOPS                 1
Ethylacetate         1
DEE                  1
Name: count, Length: 101, dtype: int64


In [3]:
print(f"Total distinct solvent strings: {df_solvent_known['solvent'].nunique()}")
print(df_solvent_known["solvent"].value_counts().to_string())

Total distinct solvent strings: 101
solvent
DMSO                                   205
dichloromethane                        204
THF                                    167
chloroform                             128
DMF                                    115
MeOH                                    88
Toluene                                 87
CH 2 Cl 2                               84
CH 2 Cl                                 81
ethanol                                 74
methanol                                63
Cyclohexane                             59
DCM                                     54
Methanol                                53
EtOH                                    52
acetonitrile                            46
Chloroform                              46
acetone                                 38
EtOAc                                   37
Ethanol                                 31
Acetonitrile                            27
Dichloromethane                         26
Water     

**built defensibly, one clear consolidation per real solvent, everything else excluded**

In [4]:
solvent_mapping = {
    # DMSO
    "DMSO": "DMSO", "Dimethylsulfoxide": "DMSO", "dimethyl sulfoxide": "DMSO", "dimethylsufoxide": "DMSO",
    # Dichloromethane
    "dichloromethane": "DCM", "CH 2 Cl 2": "DCM", "DCM": "DCM", "Dichloromethane": "DCM",
    "CH2Cl2": "DCM", "Methylene chloride": "DCM",
    # THF (kept distinct from MTHF)
    "THF": "THF", "tetrahydrofuran": "THF", "Tetrahydrofuran": "THF",
    # Chloroform (kept distinct from Tetrachloromethane)
    "chloroform": "Chloroform", "Chloroform": "Chloroform", "CHCl3": "Chloroform", "CHCl": "Chloroform",
    # DMF
    "DMF": "DMF", "dimethylformamide": "DMF", "Dimethylformamide": "DMF", "dmf": "DMF",
    # Methanol
    "MeOH": "Methanol", "methanol": "Methanol", "Methanol": "Methanol", "CH 3 OH": "Methanol",
    # Toluene
    "Toluene": "Toluene", "toluene": "Toluene",
    # Cyclohexane (kept distinct from Methylcyclohexane/MCH)
    "Cyclohexane": "Cyclohexane", "cyclohexane": "Cyclohexane", "C6H12": "Cyclohexane", "CHX": "Cyclohexane",
    # Ethanol
    "ethanol": "Ethanol", "EtOH": "Ethanol", "Ethanol": "Ethanol",
    # Acetonitrile
    "acetonitrile": "Acetonitrile", "Acetonitrile": "Acetonitrile", "MeCN": "Acetonitrile",
    "ACN": "Acetonitrile", "CH 3 CN": "Acetonitrile", "CH3CN": "Acetonitrile",
    # Acetone
    "acetone": "Acetone", "Acetone": "Acetone",
    # Water
    "Water": "Water", "water": "Water", "H2O": "Water", "H 2 O": "Water",
    # Dioxane
    "Dioxane": "Dioxane", "dioxane": "Dioxane", "1,4-dioxane": "Dioxane",
    # Hexane (kept distinct from Heptane)
    "Hexane": "Hexane", "hexane": "Hexane", "n-hexane": "Hexane",
    # Benzene
    "Benzene": "Benzene", "C6H6": "Benzene",
    # Ethyl acetate
    "EtOAc": "EthylAcetate", "Ethyl acetate": "EthylAcetate", "Ethylacetate": "EthylAcetate",
    # Chlorobenzene
    "chlorobenzene": "Chlorobenzene", "Chlorobenzene": "Chlorobenzene",
    # Single-entry but chemically clear, kept as their own categories
    "Pyridine": "Pyridine", "Heptane": "Heptane", "heptane": "Heptane",
    "MTHF": "MTHF", "NMP": "NMP", "PhCN": "PhCN", "PhCl": "PhCl",
    "Methylcyclohexane": "Methylcyclohexane", "MCH": "Methylcyclohexane",
    "Butanol": "Butanol", "Isopropanol": "Isopropanol", "propanol": "Propanol", "Propanol": "Propanol",
    "Benzyl alcohol": "BenzylAlcohol", "Tetrachloromethane": "CarbonTetrachloride",
    "propylene carbonate": "PropyleneCarbonate", "octene": "Octene", "Pentane": "Pentane",
}

# Explicitly NOT solvents -- measurement matrices/substrates/buffers, exclude these rows entirely
non_solvent_entries = [
    "quartz", "KBr", "silica", "barium sulfate", "PMMA", "PVC", "Ni", "N", "Cl", "HCl",
    "SDS", "PBS", "MOPS", "TBAF"
]

# Ambiguous/unusable -- exclude rather than guess
ambiguous_entries = [
    "CH 2 Cl", "CH3OH–H2O", "CH 2 Cl 2 or CH 2 Cl 2 / DMSO",
    "CH2Cl2 ( c 2 × 10–5 mol / l )", "10 − 4–10 − 6 M",
    "CH 2 Cl 2 + 0.2 mmol l − 1 Bu 4 NBF", "TFA", "DEE"
]

df_solvent_known["solvent_clean"] = df_solvent_known["solvent"].map(solvent_mapping)

excluded_mask = df_solvent_known["solvent"].isin(non_solvent_entries + ambiguous_entries)
print(f"Rows excluded as non-solvent or ambiguous: {excluded_mask.sum()}")

df_solvent_final = df_solvent_known[~excluded_mask & df_solvent_known["solvent_clean"].notna()].copy()

print(f"Rows before mapping: {df_solvent_known.shape[0]}")
print(f"Rows after exclusions and mapping: {df_solvent_final.shape[0]}")
print(f"Unmapped/unhandled rows (should be 0 or investigate): {df_solvent_known.shape[0] - excluded_mask.sum() - df_solvent_final.shape[0]}")
print(df_solvent_final["solvent_clean"].value_counts())

Rows excluded as non-solvent or ambiguous: 163
Rows before mapping: 2218
Rows after exclusions and mapping: 2052
Unmapped/unhandled rows (should be 0 or investigate): 3
solvent_clean
DCM                    381
DMSO                   222
Methanol               205
Chloroform             196
THF                    175
Ethanol                157
Acetonitrile           135
DMF                    123
Toluene                108
Cyclohexane             70
Acetone                 63
Water                   47
Dioxane                 43
EthylAcetate            39
Hexane                  30
Benzene                 14
MTHF                     6
Chlorobenzene            6
Heptane                  5
NMP                      5
PhCN                     4
Butanol                  2
Methylcyclohexane        2
PropyleneCarbonate       2
BenzylAlcohol            2
Isopropanol              2
Pyridine                 2
Propanol                 2
PhCl                     1
CarbonTetrachloride      1
Pentane

**Three unmapped rows**

In [5]:
unmapped_mask = df_solvent_known["solvent_clean"].isna() & ~df_solvent_known["solvent"].isin(non_solvent_entries + ambiguous_entries)
print(df_solvent_known[unmapped_mask]["solvent"].tolist())

['PGMEA', 'PGMEA', 'Hex']


In [6]:
threshold = 10
counts = df_solvent_final["solvent_clean"].value_counts()
rare_solvents = counts[counts < threshold].index.tolist()

df_solvent_final["solvent_grouped"] = df_solvent_final["solvent_clean"].apply(
    lambda x: "Other" if x in rare_solvents else x
)

print(df_solvent_final["solvent_grouped"].value_counts())
print(f"\nFinal dataset size for solvent-based analysis: {df_solvent_final.shape[0]}")

solvent_grouped
DCM             381
DMSO            222
Methanol        205
Chloroform      196
THF             175
Ethanol         157
Acetonitrile    135
DMF             123
Toluene         108
Cyclohexane      70
Acetone          63
Water            47
Other            44
Dioxane          43
EthylAcetate     39
Hexane           30
Benzene          14
Name: count, dtype: int64

Final dataset size for solvent-based analysis: 2052


In [7]:
extra_mapping = {"PGMEA": "PropyleneGlycolMethylEtherAcetate", "Hex": "Hexane"}
df_solvent_known.loc[df_solvent_known["solvent"].isin(extra_mapping.keys()), "solvent_clean"] = \
    df_solvent_known.loc[df_solvent_known["solvent"].isin(extra_mapping.keys()), "solvent"].map(extra_mapping)

# Rebuild final dataset now that these 3 rows are accounted for
df_solvent_final = df_solvent_known[
    ~df_solvent_known["solvent"].isin(non_solvent_entries + ambiguous_entries)
    & df_solvent_known["solvent_clean"].notna()
].copy()

threshold = 10
counts = df_solvent_final["solvent_clean"].value_counts()
rare_solvents = counts[counts < threshold].index.tolist()
df_solvent_final["solvent_grouped"] = df_solvent_final["solvent_clean"].apply(
    lambda x: "Other" if x in rare_solvents else x
)

print(f"Final dataset size: {df_solvent_final.shape[0]}")
print(df_solvent_final["solvent_grouped"].value_counts())

Final dataset size: 2055
solvent_grouped
DCM             381
DMSO            222
Methanol        205
Chloroform      196
THF             175
Ethanol         157
Acetonitrile    135
DMF             123
Toluene         108
Cyclohexane      70
Acetone          63
Water            47
Other            46
Dioxane          43
EthylAcetate     39
Hexane           31
Benzene          14
Name: count, dtype: int64


**merging this solvent-labeled subset with Phase 2 structural features, one-hot encode solvent, and refit both Ridge and GP — same models, same evaluation, on this smaller-but-solvent-informed dataset**

In [8]:
feature_cols = [
    "HeavyAtomCount", "LargestConjugatedSystemSize", "TPSA",
    "NumHDonors", "NumRotatableBonds", "FractionCSP3", "MolLogP_clipped",
    "NumDonorGroups", "NumAcceptorGroups", "HasPushPull"
]# Merge solvent info with structural features from Phase 2 output
df_features_full = pd.read_csv("beard_model_ready_features.csv")

df_merged = df_solvent_final.merge(
    df_features_full[["canonical_smi"] + feature_cols],
    on="canonical_smi",
    how="inner"
)

print(f"Rows after merging with structural features: {df_merged.shape[0]}")
print(f"(Expected: some loss here, since Phase 2's MW/heavy-atom scoping filter may have dropped a few of these compounds too)")

# One-hot encode the grouped solvent categories
solvent_dummies = pd.get_dummies(df_merged["solvent_grouped"], prefix="solvent")
print(f"\nSolvent dummy columns created: {solvent_dummies.shape[1]}")
print(solvent_dummies.columns.tolist())

Rows after merging with structural features: 1987
(Expected: some loss here, since Phase 2's MW/heavy-atom scoping filter may have dropped a few of these compounds too)

Solvent dummy columns created: 17
['solvent_Acetone', 'solvent_Acetonitrile', 'solvent_Benzene', 'solvent_Chloroform', 'solvent_Cyclohexane', 'solvent_DCM', 'solvent_DMF', 'solvent_DMSO', 'solvent_Dioxane', 'solvent_Ethanol', 'solvent_EthylAcetate', 'solvent_Hexane', 'solvent_Methanol', 'solvent_Other', 'solvent_THF', 'solvent_Toluene', 'solvent_Water']


**Build the combined feature matrix and refit — same models, same evaluation discipline as Phase 3/4, on this smaller but solvent-informed dataset**

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X_combined = pd.concat([df_merged[feature_cols].reset_index(drop=True), solvent_dummies.reset_index(drop=True)], axis=1)
y_combined = df_merged["lambda_max_exp_nm"].reset_index(drop=True)

print(f"Combined feature matrix shape: {X_combined.shape}")

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_combined, y_combined, test_size=0.2, random_state=42
)

print(f"Train: {X_train_c.shape}, Test: {X_test_c.shape}")

Combined feature matrix shape: (1987, 27)
Train: (1589, 27), Test: (398, 27)


**Scalling**

In [10]:
scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

**Ridge regession**

In [11]:
from sklearn.linear_model import RidgeCV

ridge_solvent = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0], cv=5)
ridge_solvent.fit(X_train_c_scaled, y_train_c)

y_pred_solvent = ridge_solvent.predict(X_test_c_scaled)
rmse_solvent = np.sqrt(mean_squared_error(y_test_c, y_pred_solvent))
r2_solvent = r2_score(y_test_c, y_pred_solvent)

print(f"Ridge + solvent (n={X_combined.shape[0]}): RMSE = {rmse_solvent:.2f} nm, R² = {r2_solvent:.3f}")
print(f"Best alpha: {ridge_solvent.alpha_}")

# Also fit Ridge WITHOUT solvent, on this SAME smaller subset -- crucial for a fair comparison
X_train_nosolv = X_train_c[feature_cols]
X_test_nosolv = X_test_c[feature_cols]
scaler_nosolv = StandardScaler()
X_train_nosolv_scaled = scaler_nosolv.fit_transform(X_train_nosolv)
X_test_nosolv_scaled = scaler_nosolv.transform(X_test_nosolv)

ridge_nosolvent = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0], cv=5)
ridge_nosolvent.fit(X_train_nosolv_scaled, y_train_c)

y_pred_nosolvent = ridge_nosolvent.predict(X_test_nosolv_scaled)
rmse_nosolvent = np.sqrt(mean_squared_error(y_test_c, y_pred_nosolvent))
r2_nosolvent = r2_score(y_test_c, y_pred_nosolvent)

print(f"Ridge, structure ONLY (same n={X_combined.shape[0]}): RMSE = {rmse_nosolvent:.2f} nm, R² = {r2_nosolvent:.3f}")

Ridge + solvent (n=1987): RMSE = 113.01 nm, R² = 0.084
Best alpha: 100.0
Ridge, structure ONLY (same n=1987): RMSE = 115.71 nm, R² = 0.040


**checking whether this modest linear improvement carries over to a GP, using the same subsampling discipline from before**

In [12]:
n_features_solvent = X_train_c_scaled.shape[1]
print(f"Feature count including solvent dummies: {n_features_solvent}")
print(f"Training set size: {X_train_c_scaled.shape[0]}")

Feature count including solvent dummies: 27
Training set size: 1589


**Gausian Processing**

In [13]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
import time

kernel_solvent = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e5))
    * RBF(length_scale=np.ones(n_features_solvent), length_scale_bounds=(1e-2, 1e4))
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e6))
)

gp_solvent = GaussianProcessRegressor(
    kernel=kernel_solvent,
    n_restarts_optimizer=2,
    random_state=42
)

start = time.time()
gp_solvent.fit(X_train_c_scaled, y_train_c)
print(f"Done in {time.time() - start:.1f} seconds.")
print(gp_solvent.kernel_)

Done in 90.1 seconds.
316**2 * RBF(length_scale=[1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1.67, 1e+04, 1, 1e+04, 101, 1e+04, 1e+04, 1e+04, 1, 1e+04, 1, 1, 1e+04, 1, 1e+04, 1e+04, 1]) + WhiteKernel(noise_level=1.37e+04)


/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better valu

In [14]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
import time

kernel_solvent = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e5))
    * RBF(length_scale=np.ones(n_features_solvent), length_scale_bounds=(1e-2, 1e4))
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e6))
)

gp_solvent = GaussianProcessRegressor(
    kernel=kernel_solvent,
    n_restarts_optimizer=2,
    random_state=42
)

start = time.time()
gp_solvent.fit(X_train_c_scaled, y_train_c)
print(f"Done in {time.time() - start:.1f} seconds.")
print(gp_solvent.kernel_)

Done in 90.6 seconds.
316**2 * RBF(length_scale=[1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1e+04, 1.67, 1e+04, 1, 1e+04, 101, 1e+04, 1e+04, 1e+04, 1, 1e+04, 1, 1, 1e+04, 1, 1e+04, 1e+04, 1]) + WhiteKernel(noise_level=1.37e+04)


/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sahanawaz/miniforge3/envs/chemo_env/lib/python3.10/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better valu

In [15]:
y_pred_gp_solvent, y_std_gp_solvent = gp_solvent.predict(X_test_c_scaled, return_std=True)

rmse_gp_solvent = np.sqrt(mean_squared_error(y_test_c, y_pred_gp_solvent))
r2_gp_solvent = r2_score(y_test_c, y_pred_gp_solvent)

print(f"GP + solvent (n={X_combined.shape[0]}): RMSE = {rmse_gp_solvent:.2f} nm, R² = {r2_gp_solvent:.3f}")
print(f"Mean predicted uncertainty: {y_std_gp_solvent.mean():.2f} nm")
print(f"Std of predicted uncertainty (variation across compounds): {y_std_gp_solvent.std():.2f} nm")

GP + solvent (n=1987): RMSE = 116.94 nm, R² = 0.019
Mean predicted uncertainty: 117.30 nm
Std of predicted uncertainty (variation across compounds): 0.82 nm


**Build a lookup table for 16 named solvent categories**

In [16]:
# Standard literature dielectric constant values (approximate, ~25°C)
# IMPORTANT: verify these against a primary reference (e.g., CRC Handbook) before using in publication
dielectric_constants = {
    "Water": 80.1,
    "DMSO": 46.7,
    "Acetonitrile": 37.5,
    "DMF": 36.7,
    "Methanol": 32.7,
    "Ethanol": 24.3,
    "Acetone": 20.7,
    "DCM": 8.93,
    "THF": 7.6,
    "EthylAcetate": 6.02,
    "Chloroform": 4.81,
    "Dioxane": 2.21,
    "Toluene": 2.38,
    "Benzene": 2.28,
    "Cyclohexane": 2.02,
    "Hexane": 1.88,
    # "Other" has no single value -- handled separately below
}

df_solvent_final["dielectric_constant"] = df_solvent_final["solvent_grouped"].map(dielectric_constants)

print(f"Rows with mapped dielectric constant: {df_solvent_final['dielectric_constant'].notna().sum()} / {df_solvent_final.shape[0]}")
print(df_solvent_final[df_solvent_final["dielectric_constant"].isna()]["solvent_grouped"].value_counts())

Rows with mapped dielectric constant: 2009 / 2055
solvent_grouped
Other    46
Name: count, dtype: int64


In [17]:
#Quick check
print("df_solvent_final defined:", "df_solvent_final" in dir())
print("solvent_mapping defined:", "solvent_mapping" in dir())
print("pd defined:", "pd" in dir())

df_solvent_final defined: True
solvent_mapping defined: True
pd defined: True


In [18]:
#Recovering the full pipeline
import pandas as pd
import numpy as np
from rdkit import Chem

col_names = [
    "SMI", "lambda1_sTDA_nm", "F1_sTDA", "lambda1_TDDFT_nm", "F1_TDDFT",
    "lambda_max_exp_nm", "extinction", "solvent"
]
df_full = pd.read_csv("paper_allDB.csv", header=None, skiprows=1, names=col_names)
df_solvent_known = df_full[df_full["solvent"].notna()].copy()

solvent_mapping = {
    "DMSO": "DMSO", "Dimethylsulfoxide": "DMSO", "dimethyl sulfoxide": "DMSO", "dimethylsufoxide": "DMSO",
    "dichloromethane": "DCM", "CH 2 Cl 2": "DCM", "DCM": "DCM", "Dichloromethane": "DCM",
    "CH2Cl2": "DCM", "Methylene chloride": "DCM",
    "THF": "THF", "tetrahydrofuran": "THF", "Tetrahydrofuran": "THF",
    "chloroform": "Chloroform", "Chloroform": "Chloroform", "CHCl3": "Chloroform", "CHCl": "Chloroform",
    "DMF": "DMF", "dimethylformamide": "DMF", "Dimethylformamide": "DMF", "dmf": "DMF",
    "MeOH": "Methanol", "methanol": "Methanol", "Methanol": "Methanol", "CH 3 OH": "Methanol",
    "Toluene": "Toluene", "toluene": "Toluene",
    "Cyclohexane": "Cyclohexane", "cyclohexane": "Cyclohexane", "C6H12": "Cyclohexane", "CHX": "Cyclohexane",
    "ethanol": "Ethanol", "EtOH": "Ethanol", "Ethanol": "Ethanol",
    "acetonitrile": "Acetonitrile", "Acetonitrile": "Acetonitrile", "MeCN": "Acetonitrile",
    "ACN": "Acetonitrile", "CH 3 CN": "Acetonitrile", "CH3CN": "Acetonitrile",
    "acetone": "Acetone", "Acetone": "Acetone",
    "Water": "Water", "water": "Water", "H2O": "Water", "H 2 O": "Water",
    "Dioxane": "Dioxane", "dioxane": "Dioxane", "1,4-dioxane": "Dioxane",
    "Hexane": "Hexane", "hexane": "Hexane", "n-hexane": "Hexane", "Hex": "Hexane",
    "Benzene": "Benzene", "C6H6": "Benzene",
    "EtOAc": "EthylAcetate", "Ethyl acetate": "EthylAcetate", "Ethylacetate": "EthylAcetate",
    "chlorobenzene": "Chlorobenzene", "Chlorobenzene": "Chlorobenzene",
    "Pyridine": "Pyridine", "Heptane": "Heptane", "heptane": "Heptane",
    "MTHF": "MTHF", "NMP": "NMP", "PhCN": "PhCN", "PhCl": "PhCl",
    "Methylcyclohexane": "Methylcyclohexane", "MCH": "Methylcyclohexane",
    "Butanol": "Butanol", "Isopropanol": "Isopropanol", "propanol": "Propanol", "Propanol": "Propanol",
    "Benzyl alcohol": "BenzylAlcohol", "Tetrachloromethane": "CarbonTetrachloride",
    "propylene carbonate": "PropyleneCarbonate", "octene": "Octene", "Pentane": "Pentane",
    "PGMEA": "PropyleneGlycolMethylEtherAcetate",
}

non_solvent_entries = ["quartz", "KBr", "silica", "barium sulfate", "PMMA", "PVC", "Ni", "N", "Cl", "HCl", "SDS", "PBS", "MOPS", "TBAF"]
ambiguous_entries = ["CH 2 Cl", "CH3OH–H2O", "CH 2 Cl 2 or CH 2 Cl 2 / DMSO", "CH2Cl2 ( c 2 × 10–5 mol / l )", "10 − 4–10 − 6 M", "CH 2 Cl 2 + 0.2 mmol l − 1 Bu 4 NBF", "TFA", "DEE"]

df_solvent_known["solvent_clean"] = df_solvent_known["solvent"].map(solvent_mapping)
excluded_mask = df_solvent_known["solvent"].isin(non_solvent_entries + ambiguous_entries)
df_solvent_final = df_solvent_known[~excluded_mask & df_solvent_known["solvent_clean"].notna()].copy()

threshold = 10
counts = df_solvent_final["solvent_clean"].value_counts()
rare_solvents = counts[counts < threshold].index.tolist()
df_solvent_final["solvent_grouped"] = df_solvent_final["solvent_clean"].apply(lambda x: "Other" if x in rare_solvents else x)

print(f"Rebuilt df_solvent_final: {df_solvent_final.shape[0]} rows (should be ~2055)")

Rebuilt df_solvent_final: 2085 rows (should be ~2055)


**Check directly which excluded strings failed to match this time**

In [19]:
# Check whether each originally-excluded string is actually still present and matching
for entry in non_solvent_entries + ambiguous_entries:
    count = (df_solvent_known["solvent"] == entry).sum()
    print(f"'{entry}': {count} matching rows")

'quartz': 16 matching rows
'KBr': 3 matching rows
'silica': 3 matching rows
'barium sulfate': 5 matching rows
'PMMA': 1 matching rows
'PVC': 2 matching rows
'Ni': 1 matching rows
'N': 1 matching rows
'Cl': 13 matching rows
'HCl': 4 matching rows
'SDS': 9 matching rows
'PBS': 3 matching rows
'MOPS': 1 matching rows
'TBAF': 1 matching rows
'CH 2 Cl': 82 matching rows
'CH3OH–H2O': 1 matching rows
'CH 2 Cl 2 or CH 2 Cl 2 / DMSO': 1 matching rows
'CH2Cl2 ( c 2 × 10–5 mol / l )': 6 matching rows
'10 − 4–10 − 6 M': 5 matching rows
'CH 2 Cl 2 + 0.2 mmol l − 1 Bu 4 NBF': 2 matching rows
'TFA': 10 matching rows
'DEE': 1 matching rows


**Check the actual exclusion count and total pool size directly, side by side**

In [20]:
print(f"Total rows with solvent present (df_solvent_known): {df_solvent_known.shape[0]}")
print(f"Rows matching exclusion criteria: {excluded_mask.sum()}")
print(f"Rows with solvent_clean successfully mapped (non-null): {df_solvent_known['solvent_clean'].notna().sum()}")
print(f"Rows with solvent_clean NULL (unmapped, not excluded): {(~excluded_mask & df_solvent_known['solvent_clean'].isna()).sum()}")

Total rows with solvent present (df_solvent_known): 2258
Rows matching exclusion criteria: 171
Rows with solvent_clean successfully mapped (non-null): 2085
Rows with solvent_clean NULL (unmapped, not excluded): 2


In [21]:
# Compare: is this the raw un-deduplicated count, vs. something filtered earlier in the original session?
print(df_full.shape[0])
print(df_full["solvent"].notna().sum())

8488
2258


**the dielectric constant mapping, on this confirmed, correct**

In [22]:
dielectric_constants = {
    "Water": 80.1, "DMSO": 46.7, "Acetonitrile": 37.5, "DMF": 36.7,
    "Methanol": 32.7, "Ethanol": 24.3, "Acetone": 20.7, "DCM": 8.93,
    "THF": 7.6, "EthylAcetate": 6.02, "Chloroform": 4.81, "Dioxane": 2.21,
    "Toluene": 2.38, "Benzene": 2.28, "Cyclohexane": 2.02, "Hexane": 1.88,
}

df_solvent_final["dielectric_constant"] = df_solvent_final["solvent_grouped"].map(dielectric_constants)

print(f"Rows with mapped dielectric constant: {df_solvent_final['dielectric_constant'].notna().sum()} / {df_solvent_final.shape[0]}")
print(df_solvent_final[df_solvent_final["dielectric_constant"].isna()]["solvent_grouped"].value_counts())

df_solvent_continuous = df_solvent_final[df_solvent_final["dielectric_constant"].notna()].copy()
print(f"\nFinal rows with continuous solvent descriptor: {df_solvent_continuous.shape[0]}")

Rows with mapped dielectric constant: 2039 / 2085
solvent_grouped
Other    46
Name: count, dtype: int64

Final rows with continuous solvent descriptor: 2039


In [23]:
from rdkit import Chem

df_solvent_continuous["mol"] = df_solvent_continuous["SMI"].apply(Chem.MolFromSmiles)
print(f"Valid SMILES: {df_solvent_continuous['mol'].notna().sum()} / {df_solvent_continuous.shape[0]}")

df_solvent_continuous = df_solvent_continuous[df_solvent_continuous["mol"].notna()].copy()
df_solvent_continuous["canonical_smi"] = df_solvent_continuous["mol"].apply(Chem.MolToSmiles)

print(f"Final rows with canonical_smi: {df_solvent_continuous.shape[0]}")

Valid SMILES: 2038 / 2039


Final rows with canonical_smi: 2038


[05:41:31] WARNING: not removing hydrogen atom without neighbors
[05:41:31] Explicit valence for atom # 4 C, 5, is greater than permitted
[05:41:31] WARNING: not removing hydrogen atom without neighbors
[05:41:31] WARNING: not removing hydrogen atom without neighbors


In [24]:
feature_cols = [
    "HeavyAtomCount", "LargestConjugatedSystemSize", "TPSA",
    "NumHDonors", "NumRotatableBonds", "FractionCSP3", "MolLogP_clipped",
    "NumDonorGroups", "NumAcceptorGroups", "HasPushPull"
]

df_features_full = pd.read_csv("beard_model_ready_features.csv")

df_merged_cont = df_solvent_continuous.merge(
    df_features_full[["canonical_smi"] + feature_cols],
    on="canonical_smi", how="inner"
)

print(f"Rows after merging with structural features: {df_merged_cont.shape[0]}")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X_cont = pd.concat([
    df_merged_cont[feature_cols].reset_index(drop=True),
    df_merged_cont[["dielectric_constant"]].reset_index(drop=True)
], axis=1)
y_cont = df_merged_cont["lambda_max_exp_nm"].reset_index(drop=True)

X_train_cont, X_test_cont, y_train_cont, y_test_cont = train_test_split(X_cont, y_cont, test_size=0.2, random_state=42)

scaler_cont = StandardScaler()
X_train_cont_scaled = scaler_cont.fit_transform(X_train_cont)
X_test_cont_scaled = scaler_cont.transform(X_test_cont)

ridge_cont = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0], cv=5)
ridge_cont.fit(X_train_cont_scaled, y_train_cont)

y_pred_cont = ridge_cont.predict(X_test_cont_scaled)
rmse_cont = np.sqrt(mean_squared_error(y_test_cont, y_pred_cont))
r2_cont = r2_score(y_test_cont, y_pred_cont)

print(f"Ridge + dielectric constant (n={X_cont.shape[0]}): RMSE = {rmse_cont:.2f} nm, R² = {r2_cont:.3f}")

# Structure-only, SAME subset, for fair comparison
X_train_nosolv2 = X_train_cont[feature_cols]
X_test_nosolv2 = X_test_cont[feature_cols]
scaler_nosolv2 = StandardScaler()
X_train_nosolv2_scaled = scaler_nosolv2.fit_transform(X_train_nosolv2)
X_test_nosolv2_scaled = scaler_nosolv2.transform(X_test_nosolv2)

ridge_nosolv2 = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0], cv=5)
ridge_nosolv2.fit(X_train_nosolv2_scaled, y_train_cont)
y_pred_nosolv2 = ridge_nosolv2.predict(X_test_nosolv2_scaled)
rmse_nosolv2 = np.sqrt(mean_squared_error(y_test_cont, y_pred_nosolv2))
r2_nosolv2 = r2_score(y_test_cont, y_pred_nosolv2)

print(f"Ridge, structure ONLY (same n={X_cont.shape[0]}): RMSE = {rmse_nosolv2:.2f} nm, R² = {r2_nosolv2:.3f}")

print(f"\nFor reference, one-hot solvent result (Section 1.19):")
print(f"Structure only (n=1987): RMSE = 115.71 nm, R² = 0.040")
print(f"Structure + one-hot solvent (n=1987): RMSE = 113.01 nm, R² = 0.084")

Rows after merging with structural features: 1953
Ridge + dielectric constant (n=1953): RMSE = 112.74 nm, R² = 0.023
Ridge, structure ONLY (same n=1953): RMSE = 112.14 nm, R² = 0.033

For reference, one-hot solvent result (Section 1.19):
Structure only (n=1987): RMSE = 115.71 nm, R² = 0.040
Structure + one-hot solvent (n=1987): RMSE = 113.01 nm, R² = 0.084


**An interaction term between polarity and electronic character, not just a standalone polarity term.**

In [25]:
df_merged_cont["dielectric_x_pushpull"] = df_merged_cont["dielectric_constant"] * df_merged_cont["HasPushPull"]

X_cont_interact = pd.concat([
    df_merged_cont[feature_cols].reset_index(drop=True),
    df_merged_cont[["dielectric_constant", "dielectric_x_pushpull"]].reset_index(drop=True)
], axis=1)

X_train_int, X_test_int, y_train_int, y_test_int = train_test_split(X_cont_interact, y_cont, test_size=0.2, random_state=42)
scaler_int = StandardScaler()
X_train_int_scaled = scaler_int.fit_transform(X_train_int)
X_test_int_scaled = scaler_int.transform(X_test_int)

ridge_int = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0], cv=5)
ridge_int.fit(X_train_int_scaled, y_train_int)
y_pred_int = ridge_int.predict(X_test_int_scaled)
rmse_int = np.sqrt(mean_squared_error(y_test_int, y_pred_int))
r2_int = r2_score(y_test_int, y_pred_int)

print(f"Ridge + dielectric + (dielectric × push-pull) interaction: RMSE = {rmse_int:.2f} nm, R² = {r2_int:.3f}")

Ridge + dielectric + (dielectric × push-pull) interaction: RMSE = 112.92 nm, R² = 0.020


**ET(30) mapping**

In [26]:
# Reichardt's ET(30) values (kcal/mol), approximate literature values
# IMPORTANT: verify against Reichardt's original compiled tables before publication
et30_values = {
    "Water": 63.1, "Methanol": 55.4, "Ethanol": 51.9, "Acetonitrile": 45.6,
    "DMSO": 45.1, "DMF": 43.2, "Acetone": 42.2, "DCM": 40.7,
    "Chloroform": 39.1, "EthylAcetate": 38.1, "THF": 37.4, "Dioxane": 36.0,
    "Benzene": 34.3, "Toluene": 33.9, "Hexane": 31.0, "Cyclohexane": 30.9,
}

df_solvent_final["et30"] = df_solvent_final["solvent_grouped"].map(et30_values)
print(f"Rows with mapped ET(30): {df_solvent_final['et30'].notna().sum()} / {df_solvent_final.shape[0]}")

Rows with mapped ET(30): 2039 / 2085


**Save this to a CSV**

In [27]:
from rdkit import Chem

df_solvent_final["mol"] = df_solvent_final["SMI"].apply(Chem.MolFromSmiles)
print(f"Valid SMILES: {df_solvent_final['mol'].notna().sum()} / {df_solvent_final.shape[0]}")

df_solvent_final = df_solvent_final[df_solvent_final["mol"].notna()].copy()
df_solvent_final["canonical_smi"] = df_solvent_final["mol"].apply(Chem.MolToSmiles)

print(f"Final rows with canonical_smi: {df_solvent_final.shape[0]}")

solvent_export_cols = ["canonical_smi", "solvent_grouped", "et30", "lambda_max_exp_nm"]
df_solvent_final[solvent_export_cols].to_csv("solvent_et30_data.csv", index=False)
print(f"Saved {df_solvent_final.shape[0]} rows to solvent_et30_data.csv")

[05:41:31] WARNING: not removing hydrogen atom without neighbors
[05:41:31] Explicit valence for atom # 4 C, 5, is greater than permitted
[05:41:31] WARNING: not removing hydrogen atom without neighbors
[05:41:31] WARNING: not removing hydrogen atom without neighbors
[05:41:31] WARNING: not removing hydrogen atom without neighbors


Valid SMILES: 2084 / 2085
Final rows with canonical_smi: 2084


Saved 2084 rows to solvent_et30_data.csv
